# IDM-VTON API Server (via HuggingFace Spaces)

Lightweight server that proxies IDM-VTON through the official HuggingFace Spaces demo.
No large model download needed — works on T4.

**How to use:**
1. Run all cells (takes ~1 min)
2. Copy the `https://xxxxx.gradio.live` URL
3. Set it as `IDM_VTON_COLAB_URL` secret in Supabase

In [ ]:
#@title 1. Install Dependencies (~30s)
!pip install -q gradio gradio_client Pillow requests
print('Dependencies installed!')

In [ ]:
#@title 2. Define try-on function (uses HF Spaces API)
from gradio_client import Client, handle_file
from PIL import Image
import requests
import tempfile
import base64
import json
import io
import os

# Connect to IDM-VTON HuggingFace Space
print('Connecting to IDM-VTON HuggingFace Space...')
hf_client = Client('yisol/IDM-VTON')
print('Connected!')

def download_to_temp(url):
    """Download image from URL to temp file."""
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    img = Image.open(io.BytesIO(resp.content)).convert('RGB')
    # Resize to reasonable dimensions
    img = img.resize((768, 1024), Image.LANCZOS)
    path = tempfile.mktemp(suffix='.jpg')
    img.save(path, 'JPEG', quality=90)
    return path

def virtual_try_on(person_url, garment_url, description='clothing item'):
    """Call IDM-VTON via HuggingFace Spaces."""
    try:
        print(f'Downloading person image...')
        person_path = download_to_temp(person_url)
        print(f'Downloading garment image...')
        garment_path = download_to_temp(garment_url)
        
        print(f'Calling IDM-VTON (description: {description})...')
        result = hf_client.predict(
            dict={"background": handle_file(person_path), "layers": [], "composite": None},
            garm_img=handle_file(garment_path),
            garment_des=description,
            is_checked=True,
            is_checked_crop=False,
            denoise_steps=30,
            seed=42,
            api_name="/tryon"
        )
        
        # Result is a tuple — first element is the output image path
        output_path = result[0]
        print(f'Result received: {output_path}')
        
        # Convert to base64
        img = Image.open(output_path).convert('RGB')
        buf = io.BytesIO()
        img.save(buf, format='JPEG', quality=90)
        b64 = base64.b64encode(buf.getvalue()).decode()
        
        # Cleanup
        os.unlink(person_path)
        os.unlink(garment_path)
        
        return json.dumps({"success": True, "image_base64": b64})
    
    except Exception as e:
        import traceback
        traceback.print_exc()
        return json.dumps({"success": False, "error": str(e)})

print('Try-on function ready!')

In [ ]:
#@title 3. Launch Gradio API Server
import gradio as gr

demo = gr.Interface(
    fn=virtual_try_on,
    inputs=[
        gr.Textbox(label='Person Image URL', placeholder='https://...signed-url...'),
        gr.Textbox(label='Garment Image URL', placeholder='https://...signed-url...'),
        gr.Textbox(label='Description', value='clothing item'),
    ],
    outputs=gr.Textbox(label='Result JSON'),
    title='IDM-VTON API Server',
    description='Virtual Try-On API via HuggingFace Spaces. Send person + garment image URLs.',
    api_name='tryon',
)

print('\n' + '='*60)
print('STARTING IDM-VTON SERVER...')
print('Copy the public URL below and set it as')
print('IDM_VTON_COLAB_URL in your Supabase secrets.')
print('='*60 + '\n')

demo.launch(share=True, quiet=False)